# Phase 1: seed variance of the baseline recipe (Colab T4)

Retrains YOLO11n and YOLO11s with seeds 0, 1, 2 using the training notebook's recipe, then evaluates each with the same command as the committed baseline.
Runtime: GPU (T4). Each run is 50 epochs; if a session will not hold all six, set `CONFIGS` to a subset and run the rest in another session.

Rules kept here: the test split is only evaluated, never trained or tuned on; the operating confidence is chosen on val by `ppe.eval`; nothing in this notebook edits results by hand.

In [ ]:
!nvidia-smi
BRANCH = "claude/phase1-honest-baseline-ofau1o"   # change to main once merged
!git clone --branch {BRANCH} https://github.com/omgawande1523/PPE.git
%cd PPE
!git log --oneline -1

In [ ]:
# Pinned versions. Colab may print dependency warnings for packages this repo does not use.
!pip install -q -r requirements.txt
import ultralytics, torch; print(ultralytics.__version__, torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Downloads the Ultralytics release and refuses to extract it if the sha256 differs.
!python -m ppe.data --download construction-ppe
!python -m ppe.golden --verify golden/golden_v1.csv

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"
CONFIGS = [
    "p1_yolo11n_seed0", "p1_yolo11n_seed1", "p1_yolo11n_seed2",
    "p1_yolo11s_seed0", "p1_yolo11s_seed1", "p1_yolo11s_seed2",
]

In [ ]:
# Train, then evaluate on val + test with the committed eval config (CPU, like the baseline rows).
for name in CONFIGS:
    model = name.split("_")[1]
    !python -m ppe.train --config configs/train/{name}.yaml
    !python -m ppe.eval --config configs/eval/baseline_yolo11s_test.yaml --weights runs/train/{name}/weights/best.pt --run-name {name} --model {model}

In [ ]:
# Keep the outputs: results.csv rows, figures and weights go to Drive.
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/PPE_phase1"
!mkdir -p {OUT}/weights
!cp results/results.csv {OUT}/results_colab.csv
!cp -r results/figures {OUT}/
for name in CONFIGS:
    !cp runs/train/{name}/weights/best.pt {OUT}/weights/{name}_best.pt
    !cp runs/train/{name}/results.csv {OUT}/{name}_train_log.csv
print("Saved to", OUT)
print("Bring results_colab.csv rows into the repo's results/results.csv (append only the new run_ids) and commit.")